# 04 - Valores atípicos y consistencia cuantitativa

**Objetivo.** Detectar valores imposibles, inconsistentes o extremos en variables cuantitativas relevantes. En esta etapa no se eliminan automáticamente outliers plausibles: se crean flags para sostener decisiones posteriores.

En real estate, un valor extremo puede ser válido. Por eso se distinguen valores imposibles de extremos plausibles.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

## Carga de datasets con flags de nulos

Si todavía no existen, se cargan los datasets preprocesados del notebook 02.

In [ ]:
base_sources = {
    'df_preprocesado_meli_alq': PROCESSED_DIR / 'df_preprocesado_meli_alq_con_flags_nulos_v1.csv',
    'df_preprocesado_meli_alq_temp': PROCESSED_DIR / 'df_preprocesado_meli_alq_temp_con_flags_nulos_v1.csv',
    'df_preprocesado_meli_vtas': PROCESSED_DIR / 'df_preprocesado_meli_vtas_con_flags_nulos_v1.csv',
    'df_preprocesado_argenprop': PROCESSED_DIR / 'df_preprocesado_argenprop_con_flags_nulos_v1.csv',
    'df_preprocesado_zonaprop': PROCESSED_DIR / 'df_preprocesado_zonaprop_con_flags_nulos_v1.csv',
    'df_preprocesado_airbnb_temp': PROCESSED_DIR / 'df_preprocesado_airbnb_temp_con_flags_nulos_v1.csv',
}
fallback_sources = {
    k: PROCESSED_DIR / f'{k}_v1.csv'
    for k in base_sources
}
preprocesados_outliers = {}
for name, path in base_sources.items():
    source = path if path.exists() else fallback_sources[name]
    if not source.exists():
        raise FileNotFoundError(f'Falta {source}. Ejecutar notebooks anteriores.')
    preprocesados_outliers[name] = pd.read_csv(source, low_memory=False)
    print(name, preprocesados_outliers[name].shape, 'desde', source.name)

## Funciones de detección de outliers

In [ ]:
VARIABLES_OUTLIERS = ['precio', 'precio_m2_ref', 'precio_noche_estimado', 'precio_mensual_estimado', 'superficie_ref_m2', 'ambientes_ref', 'dormitorios_ref', 'banios_ref', 'altura', 'latitud', 'longitud', 'rating', 'reviews']

CABA_LAT_MIN, CABA_LAT_MAX = -34.75, -34.50
CABA_LON_MIN, CABA_LON_MAX = -58.55, -58.30


def add_outlier_flags(df, dataset_name, p_low=0.01, p_high=0.99):
    out = df.copy()
    rows = []
    group_cols = [c for c in ['fuente_norm', 'tipo_operacion_norm', 'moneda_norm'] if c in out.columns]

    for col in VARIABLES_OUTLIERS:
        if col not in out.columns or not pd.api.types.is_numeric_dtype(out[col]):
            continue
        s = out[col]

        invalid_flag = f'{col}_valor_invalido_flag'
        if col in ['precio', 'precio_m2_ref', 'precio_noche_estimado', 'precio_mensual_estimado', 'superficie_ref_m2', 'ambientes_ref', 'dormitorios_ref', 'banios_ref']:
            out[invalid_flag] = (s.notna() & (s <= 0)).astype('int8')
        elif col == 'rating':
            out[invalid_flag] = (s.notna() & ((s < 0) | (s > 5))).astype('int8')
        elif col == 'latitud':
            out[invalid_flag] = (s.notna() & ((s < CABA_LAT_MIN) | (s > CABA_LAT_MAX))).astype('int8')
        elif col == 'longitud':
            out[invalid_flag] = (s.notna() & ((s < CABA_LON_MIN) | (s > CABA_LON_MAX))).astype('int8')
        else:
            out[invalid_flag] = 0

        valid = out[out[invalid_flag].eq(0) & out[col].notna()].copy()
        outlier_flag = f'{col}_outlier_p01_p99_flag'
        out[outlier_flag] = 0

        if valid.empty or valid[col].nunique(dropna=True) <= 1:
            continue

        if group_cols:
            qs = valid.groupby(group_cols)[col].quantile([p_low, p_high]).unstack()
            qs.columns = ['p01', 'p99']
            valid = valid.join(qs, on=group_cols)
            mask_outlier = (valid[col] < valid['p01']) | (valid[col] > valid['p99'])
            out.loc[valid.index, outlier_flag] = mask_outlier.astype('int8')
        else:
            q_low, q_high = valid[col].quantile([p_low, p_high])
            mask_outlier = (valid[col] < q_low) | (valid[col] > q_high)
            out.loc[valid.index, outlier_flag] = mask_outlier.astype('int8')

        rows.append({
            'dataset': dataset_name,
            'variable': col,
            'n_no_nulos': int(s.notna().sum()),
            'n_invalidos': int(out[invalid_flag].sum()),
            'n_outliers_p01_p99': int(out[outlier_flag].sum()),
            'pct_invalidos': round(out[invalid_flag].mean() * 100, 2),
            'pct_outliers_p01_p99': round(out[outlier_flag].mean() * 100, 2),
        })

    return out, pd.DataFrame(rows)

## Aplicación por dataset

In [ ]:
preprocesados_con_outliers = {}
reportes_outliers = []
for name, df in preprocesados_outliers.items():
    df_out, reporte = add_outlier_flags(df, name)
    preprocesados_con_outliers[name] = df_out
    if not reporte.empty:
        reportes_outliers.append(reporte)
    print(name, df_out.shape, 'flags outliers:', sum(c.endswith('_flag') for c in df_out.columns))

reporte_outliers = pd.concat(reportes_outliers, ignore_index=True) if reportes_outliers else pd.DataFrame()
display(reporte_outliers)
reporte_outliers.to_csv(REPORTS_DIR / 'reporte_outliers_v1.csv', index=False)

## Celdas de decisión: tratamiento de valores inválidos y outliers

La estrategia por defecto es **marcar** valores inválidos y extremos. No se eliminan outliers plausibles porque en el mercado inmobiliario pueden representar propiedades reales. Si se decide limpiar de forma activa, se recomienda transformar a nulo solo valores imposibles y documentar el impacto.

In [ ]:
# Parametros conservadores.
NULIFICAR_VALORES_INVALIDOS = False
ELIMINAR_OUTLIERS_P01_P99 = False

resumen_decisiones_outliers = []
for name, df in preprocesados_con_outliers.items():
    filas_antes = len(df)
    invalid_flags = [c for c in df.columns if c.endswith('_valor_invalido_flag')]
    outlier_flags = [c for c in df.columns if c.endswith('_outlier_p01_p99_flag')]

    filas_con_invalido = int(df[invalid_flags].any(axis=1).sum()) if invalid_flags else 0
    filas_con_outlier = int(df[outlier_flags].any(axis=1).sum()) if outlier_flags else 0

    if NULIFICAR_VALORES_INVALIDOS:
        for flag_col in invalid_flags:
            base_col = flag_col.replace('_valor_invalido_flag', '')
            if base_col in df.columns:
                df.loc[df[flag_col].eq(1), base_col] = np.nan

    if ELIMINAR_OUTLIERS_P01_P99 and outlier_flags:
        df = df.loc[~df[outlier_flags].any(axis=1)].copy()

    preprocesados_con_outliers[name] = df
    resumen_decisiones_outliers.append({
        'dataset': name,
        'filas_antes': filas_antes,
        'filas_con_valor_invalido': filas_con_invalido,
        'filas_con_outlier_p01_p99': filas_con_outlier,
        'nulificar_valores_invalidos': NULIFICAR_VALORES_INVALIDOS,
        'eliminar_outliers_p01_p99': ELIMINAR_OUTLIERS_P01_P99,
        'filas_despues': len(df),
    })

resumen_decisiones_outliers = pd.DataFrame(resumen_decisiones_outliers)
display(resumen_decisiones_outliers)
resumen_decisiones_outliers.to_csv(REPORTS_DIR / 'decisiones_tratamiento_outliers_v1.csv', index=False)

## Visualización resumida de outliers

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    import matplotlib.pyplot as plt
    HAS_SEABORN = False

if not reporte_outliers.empty:
    pivot_outliers = reporte_outliers.pivot_table(index=['dataset', 'variable'], values='pct_outliers_p01_p99', aggfunc='mean').sort_values('pct_outliers_p01_p99', ascending=False)
    display(pivot_outliers.head(40))

    top_plot = pivot_outliers.head(30).reset_index()
    plt.figure(figsize=(12, max(5, len(top_plot) * 0.28)))
    labels = top_plot['dataset'] + ' | ' + top_plot['variable']
    plt.barh(labels, top_plot['pct_outliers_p01_p99'])
    plt.xlabel('% outliers p01-p99')
    plt.title('Variables con mayor proporcion de valores extremos')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## Guardado de versiones con flags de outliers

In [ ]:
for name, df in preprocesados_con_outliers.items():
    path = PROCESSED_DIR / f'{name}_con_outliers_v1.csv'
    df.to_csv(path, index=False)
    print('Guardado:', path.relative_to(REPO_ROOT), df.shape)

canonical_cols = [
    'id_publicacion', 'fuente_norm', 'fuente', 'tipo_operacion_norm', 'tipo_operacion', 'url', 'fecha_extraccion',
    'scraped_at_utc', '_archivo_origen', 'titulo', 'descripcion', 'tipo_propiedad', 'moneda_norm', 'moneda',
    'precio', 'precio_texto', 'precio_m2_ref', 'precio_noche_estimado', 'precio_mensual_estimado', 'barrio',
    'barrio_norm', 'comuna', 'localidad', 'provincia', 'pais', 'direccion', 'direccion_completa', 'calle',
    'altura', 'latitud', 'longitud', 'superficie_ref_m2', 'ambientes_ref', 'dormitorios_ref', 'banios_ref'
]
frames = []
for name, df in preprocesados_con_outliers.items():
    cols = [c for c in df.columns if c in canonical_cols or c.endswith('_is_null') or c.endswith('_flag')]
    tmp = df[cols].copy()
    tmp['dataset_preprocesado'] = name
    frames.append(tmp)

df_publicaciones_consolidadas_outliers = pd.concat(frames, ignore_index=True, sort=False)
df_publicaciones_consolidadas_outliers.to_csv(PROCESSED_DIR / 'df_publicaciones_consolidadas_con_outliers_v1.csv', index=False)
print(df_publicaciones_consolidadas_outliers.shape)

## Criterio de cierre

No se eliminan automáticamente outliers plausibles. Los valores imposibles quedan identificados con flags y las variables extremas se marcan por percentiles dentro de fuente/operación/moneda cuando esas columnas existen.